# Tool Calling Evaluation Notebook v2

This notebook evaluates the precision of tool calling by an LLM agent with 6 different tools.
We'll test various scenarios and measure how accurately the agent selects the correct tool(s).

**v2 Enhancements:**
- Tool Invocation Precision & Recall
- Average Tool Calls per Task
- Tool Success Rate
- Tool-Attributable Task Success
- Cost per Successful Task

**Google Colab Ready** - Just run the cells in order!

## 0. Install Dependencies (Colab)

In [ ]:
# Install required packages
# Note: We upgrade numpy first to avoid binary incompatibility issues
!pip install -q --upgrade numpy
!pip install -q langchain==0.2.14 langchain-groq==0.1.9 matplotlib pandas

# If you see numpy dtype errors, restart the runtime after this cell:
# Runtime > Restart runtime (in Colab menu)

## 1. Setup and API Key Configuration

In [ ]:
import os
import math
import time
from datetime import datetime
from typing import List, Dict, Any, Optional, Tuple
from getpass import getpass
from dataclasses import dataclass, field

# Set up Groq API key
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")

print("API key configured!")

In [ ]:
import pandas as pd
from langchain_groq import ChatGroq
from langchain.tools import tool
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain.prompts import ChatPromptTemplate

print("Imports complete!")

## 2. Define 6 Tools

We create tools for different domains:
1. **Calculator** - Basic arithmetic operations
2. **Square Root** - Compute square roots
3. **Temperature Converter** - Convert between Celsius and Fahrenheit
4. **String Reverser** - Reverse a string
5. **Word Counter** - Count words in text
6. **Date Info** - Get current date/time information

In [ ]:
@tool
def calculator(expression: str) -> str:
    """Evaluate a basic arithmetic expression. Supports +, -, *, /, and parentheses.
    Use this for addition, subtraction, multiplication, and division.
    Example: '2 + 3 * 4' or '(10 - 5) / 2'
    """
    try:
        # Only allow safe characters for math
        allowed = set('0123456789+-*/().% ')
        if not all(c in allowed for c in expression):
            return "Error: Invalid characters in expression"
        result = eval(expression)
        return f"Result: {result}"
    except Exception as e:
        return f"Error evaluating expression: {str(e)}"


@tool
def square_root(number: float) -> str:
    """Calculate the square root of a number. Use this specifically for square root calculations.
    Example: square_root(16) returns 4.0
    """
    if number < 0:
        return "Error: Cannot calculate square root of negative number"
    result = math.sqrt(number)
    return f"The square root of {number} is {result}"


@tool
def temperature_converter(value: float, from_unit: str) -> str:
    """Convert temperature between Celsius and Fahrenheit.
    from_unit should be 'C' for Celsius or 'F' for Fahrenheit.
    Example: temperature_converter(100, 'C') converts 100C to Fahrenheit.
    """
    from_unit = from_unit.upper()
    if from_unit == 'C':
        result = (value * 9/5) + 32
        return f"{value}C = {result}F"
    elif from_unit == 'F':
        result = (value - 32) * 5/9
        return f"{value}F = {result}C"
    else:
        return "Error: Use 'C' for Celsius or 'F' for Fahrenheit"


@tool
def string_reverser(text: str) -> str:
    """Reverse a string. Use this when asked to reverse text or spell something backwards.
    Example: string_reverser('hello') returns 'olleh'
    """
    return f"Reversed: {text[::-1]}"


@tool
def word_counter(text: str) -> str:
    """Count the number of words in a text. Use this for word counting tasks.
    Example: word_counter('Hello world') returns 2
    """
    words = text.split()
    return f"Word count: {len(words)}"


@tool
def date_info(query: str) -> str:
    """Get current date and time information. Use for questions about today's date, current time, day of week, etc.
    Query can be: 'date', 'time', 'day', 'full'
    """
    now = datetime.now()
    query = query.lower()
    if 'date' in query:
        return f"Today's date: {now.strftime('%Y-%m-%d')}"
    elif 'time' in query:
        return f"Current time: {now.strftime('%H:%M:%S')}"
    elif 'day' in query:
        return f"Today is: {now.strftime('%A')}"
    else:
        return f"Current datetime: {now.strftime('%Y-%m-%d %H:%M:%S')} ({now.strftime('%A')})"


# List all tools
tools = [calculator, square_root, temperature_converter, string_reverser, word_counter, date_info]
TOOL_NAMES = [t.name for t in tools]

print(f"Created {len(tools)} tools:")
for t in tools:
    print(f"  - {t.name}: {t.description[:60]}...")

## 3. Create the Agent

In [ ]:
# Initialize the LLM
llm = ChatGroq(
    model="llama-3.1-70b-versatile",
    temperature=0
)

# Create the prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful assistant with access to various tools.
Use the appropriate tool(s) to answer user questions.
Always use tools when they are relevant to the question.
Be precise in selecting the right tool for each task."""),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}")
])

# Create the agent
agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True, return_intermediate_steps=True)

print("Agent created successfully!")

## 4. Define Test Scenarios

We create test cases with:
- **question**: The user query
- **expected_tools**: List of tools that should be called
- **category**: Type of scenario (single tool, multi-tool, no tool, ambiguous)
- **requires_external_data**: Whether task depends on tool output (for Tool-Attributable Success)

In [ ]:
test_scenarios = [
    # Single tool scenarios
    {
        "question": "What is 25 + 17?",
        "expected_tools": ["calculator"],
        "category": "single_tool",
        "requires_external_data": True,
        "optimal_tool_count": 1
    },
    {
        "question": "What is the square root of 144?",
        "expected_tools": ["square_root"],
        "category": "single_tool",
        "requires_external_data": True,
        "optimal_tool_count": 1
    },
    {
        "question": "Convert 30 degrees Celsius to Fahrenheit",
        "expected_tools": ["temperature_converter"],
        "category": "single_tool",
        "requires_external_data": True,
        "optimal_tool_count": 1
    },
    {
        "question": "Reverse the word 'python'",
        "expected_tools": ["string_reverser"],
        "category": "single_tool",
        "requires_external_data": True,
        "optimal_tool_count": 1
    },
    {
        "question": "How many words are in 'The quick brown fox jumps over the lazy dog'?",
        "expected_tools": ["word_counter"],
        "category": "single_tool",
        "requires_external_data": True,
        "optimal_tool_count": 1
    },
    {
        "question": "What day of the week is today?",
        "expected_tools": ["date_info"],
        "category": "single_tool",
        "requires_external_data": True,
        "optimal_tool_count": 1
    },
    
    # Multi-tool scenarios
    {
        "question": "Calculate 15 * 4 and also tell me the square root of the result",
        "expected_tools": ["calculator", "square_root"],
        "category": "multi_tool",
        "requires_external_data": True,
        "optimal_tool_count": 2
    },
    {
        "question": "What is today's date and what is 100 + 200?",
        "expected_tools": ["date_info", "calculator"],
        "category": "multi_tool",
        "requires_external_data": True,
        "optimal_tool_count": 2
    },
    
    # Potentially ambiguous scenarios
    {
        "question": "What is 64 to the power of 0.5?",
        "expected_tools": ["calculator"],
        "category": "ambiguous",
        "acceptable_alternatives": [["square_root"]],
        "requires_external_data": True,
        "optimal_tool_count": 1
    },
    {
        "question": "Tell me about the current moment",
        "expected_tools": ["date_info"],
        "category": "ambiguous",
        "requires_external_data": True,
        "optimal_tool_count": 1
    },
    
    # No tool needed scenarios
    {
        "question": "What is the capital of France?",
        "expected_tools": [],
        "category": "no_tool",
        "requires_external_data": False,
        "optimal_tool_count": 0
    },
    {
        "question": "Explain what machine learning is",
        "expected_tools": [],
        "category": "no_tool",
        "requires_external_data": False,
        "optimal_tool_count": 0
    }
]

print(f"Defined {len(test_scenarios)} test scenarios:")
for i, s in enumerate(test_scenarios, 1):
    print(f"  {i}. [{s['category']}] {s['question'][:50]}...")

## 5. Enhanced Data Structures for v2 Metrics

We extend the evaluation to capture additional information needed for the new metrics:
- Tool call success/failure status
- Execution time and cost estimates
- Retry counts

In [ ]:
@dataclass
class ToolCallLog:
    """Detailed log for a single tool call."""
    tool_name: str
    parameters: Dict[str, Any]
    output: str
    success: bool  # True if tool executed without error
    is_correct: bool  # True if this was an expected tool for the task
    execution_time_ms: float = 0.0
    retry_count: int = 0
    
    # Cost estimation (in arbitrary units)
    # Assumes: tool execution has a base cost
    TOOL_COST_PER_CALL = 0.001  # $0.001 per tool call
    
    @property
    def cost(self) -> float:
        return self.TOOL_COST_PER_CALL * (1 + self.retry_count)


@dataclass 
class TaskEvaluation:
    """Complete evaluation result for a single task."""
    question: str
    category: str
    expected_tools: List[str]
    actual_tools: List[str]
    tool_call_logs: List[ToolCallLog]
    output: Optional[str]
    error: Optional[str]
    requires_external_data: bool
    optimal_tool_count: int
    acceptable_alternatives: List[List[str]] = field(default_factory=list)
    
    # Token/cost tracking
    input_tokens: int = 0
    output_tokens: int = 0
    
    # Cost constants (Groq pricing approximation)
    COST_PER_1K_INPUT_TOKENS = 0.0005
    COST_PER_1K_OUTPUT_TOKENS = 0.001
    
    @property
    def task_success(self) -> bool:
        """Heuristic: task is successful if no error and output exists."""
        return self.error is None and self.output is not None
    
    @property
    def llm_cost(self) -> float:
        """Estimate LLM API cost."""
        input_cost = (self.input_tokens / 1000) * self.COST_PER_1K_INPUT_TOKENS
        output_cost = (self.output_tokens / 1000) * self.COST_PER_1K_OUTPUT_TOKENS
        return input_cost + output_cost
    
    @property
    def tool_cost(self) -> float:
        """Sum of all tool execution costs."""
        return sum(log.cost for log in self.tool_call_logs)
    
    @property
    def total_cost(self) -> float:
        """Total cost = LLM + tool costs."""
        return self.llm_cost + self.tool_cost


print("Enhanced data structures defined!")

## 6. Evaluation Helper Functions

In [ ]:
def extract_tool_calls_detailed(intermediate_steps: List, expected_tools: List[str]) -> List[ToolCallLog]:
    """Extract detailed tool call information from agent's intermediate steps."""
    tool_logs = []
    expected_set = set(expected_tools)
    
    for step in intermediate_steps:
        action = step[0]
        output = step[1] if len(step) > 1 else ""
        
        if hasattr(action, 'tool'):
            tool_name = action.tool
            tool_input = action.tool_input if hasattr(action, 'tool_input') else {}
            
            # Determine if tool call was successful (heuristic: no "Error" in output)
            output_str = str(output)
            success = "Error" not in output_str and "error" not in output_str.lower()
            
            # Determine if this was a correct tool to call
            is_correct = tool_name in expected_set
            
            log = ToolCallLog(
                tool_name=tool_name,
                parameters=tool_input if isinstance(tool_input, dict) else {"input": tool_input},
                output=output_str,
                success=success,
                is_correct=is_correct,
                execution_time_ms=0.0,  # Not tracked by default
                retry_count=0  # Would need custom tracking
            )
            tool_logs.append(log)
    
    return tool_logs


def estimate_tokens(text: str) -> int:
    """Rough token estimation (4 chars per token average)."""
    return len(text) // 4


def check_tool_output_in_response(tool_logs: List[ToolCallLog], response: str) -> bool:
    """Heuristic: Check if any tool output appears in the final response."""
    if not response or not tool_logs:
        return False
    
    response_lower = response.lower()
    for log in tool_logs:
        # Extract key parts of tool output to check
        output_parts = log.output.replace("Result:", "").replace("The square root of", "").strip()
        # Check if numeric results appear in response
        for part in output_parts.split():
            try:
                # If it's a number, check if it appears
                float(part.replace(",", ""))
                if part in response:
                    return True
            except ValueError:
                # Check for string matches
                if len(part) > 3 and part.lower() in response_lower:
                    return True
    return False


print("Helper functions defined!")

## 7. Run Enhanced Evaluation

In [ ]:
def run_enhanced_evaluation(scenarios: List[Dict], agent_executor: AgentExecutor) -> List[TaskEvaluation]:
    """Run all test scenarios with enhanced logging for v2 metrics."""
    results = []
    
    for i, scenario in enumerate(scenarios, 1):
        print(f"\n{'='*60}")
        print(f"Test {i}/{len(scenarios)}: {scenario['question']}")
        print(f"Expected tools: {scenario['expected_tools']}")
        print(f"{'='*60}")
        
        try:
            # Run the agent
            start_time = time.time()
            response = agent_executor.invoke({"input": scenario["question"]})
            elapsed_ms = (time.time() - start_time) * 1000
            
            # Extract detailed tool calls
            tool_logs = extract_tool_calls_detailed(
                response.get("intermediate_steps", []),
                scenario["expected_tools"]
            )
            
            # Get tool names for backward compatibility
            tools_called = [log.tool_name for log in tool_logs]
            output = response.get("output", "")
            
            # Estimate tokens
            input_tokens = estimate_tokens(scenario["question"])
            output_tokens = estimate_tokens(output) if output else 0
            
            result = TaskEvaluation(
                question=scenario["question"],
                category=scenario["category"],
                expected_tools=scenario["expected_tools"],
                actual_tools=tools_called,
                tool_call_logs=tool_logs,
                output=output,
                error=None,
                requires_external_data=scenario.get("requires_external_data", True),
                optimal_tool_count=scenario.get("optimal_tool_count", len(scenario["expected_tools"])),
                acceptable_alternatives=scenario.get("acceptable_alternatives", []),
                input_tokens=input_tokens,
                output_tokens=output_tokens
            )
            
            print(f"\nTools called: {tools_called}")
            print(f"Execution time: {elapsed_ms:.1f}ms")
            
        except Exception as e:
            result = TaskEvaluation(
                question=scenario["question"],
                category=scenario["category"],
                expected_tools=scenario["expected_tools"],
                actual_tools=[],
                tool_call_logs=[],
                output=None,
                error=str(e),
                requires_external_data=scenario.get("requires_external_data", True),
                optimal_tool_count=scenario.get("optimal_tool_count", len(scenario["expected_tools"])),
                acceptable_alternatives=scenario.get("acceptable_alternatives", [])
            )
            print(f"Error: {e}")
        
        results.append(result)
    
    return results

# Run the evaluation
print("Starting enhanced evaluation...\n")
evaluation_results = run_enhanced_evaluation(test_scenarios, agent_executor)

---

# NEW METRICS (v2)

The following sections add the new evaluation metrics.

## 8. Tool Invocation Precision

### Definition
**Tool Invocation Precision** measures the proportion of tool calls that were correct (i.e., the tool was expected for the given task).

### Formula
```
Precision = correct_tool_calls / total_tool_calls
```

A tool call is considered "correct" if:
1. The tool name matches one of the expected tools for the task
2. OR the tool is in the acceptable alternatives list

In [ ]:
def compute_tool_invocation_precision(results: List[TaskEvaluation]) -> Dict[str, Any]:
    """
    Compute Tool Invocation Precision across all tasks.
    
    Precision = correct_tool_calls / total_tool_calls
    
    A tool call is "correct" if it matches an expected tool or acceptable alternative.
    """
    total_tool_calls = 0
    correct_tool_calls = 0
    per_task_precision = []
    
    for result in results:
        # Build set of acceptable tools (expected + alternatives)
        acceptable_tools = set(result.expected_tools)
        for alt in result.acceptable_alternatives:
            acceptable_tools.update(alt)
        
        task_total = len(result.actual_tools)
        task_correct = sum(1 for t in result.actual_tools if t in acceptable_tools)
        
        total_tool_calls += task_total
        correct_tool_calls += task_correct
        
        # Per-task precision
        if task_total > 0:
            per_task_precision.append(task_correct / task_total)
        else:
            # No tools called - precision is 1.0 if none expected, 0.0 otherwise
            per_task_precision.append(1.0 if len(result.expected_tools) == 0 else 0.0)
    
    # Overall precision
    overall_precision = correct_tool_calls / total_tool_calls if total_tool_calls > 0 else 1.0
    avg_task_precision = sum(per_task_precision) / len(per_task_precision) if per_task_precision else 0.0
    
    return {
        "overall_precision": overall_precision,
        "avg_task_precision": avg_task_precision,
        "total_tool_calls": total_tool_calls,
        "correct_tool_calls": correct_tool_calls,
        "per_task_precision": per_task_precision
    }


# Compute and display
precision_metrics = compute_tool_invocation_precision(evaluation_results)

print("=" * 50)
print("TOOL INVOCATION PRECISION")
print("=" * 50)
print(f"Overall Precision:     {precision_metrics['overall_precision']:.3f}")
print(f"Avg Task Precision:    {precision_metrics['avg_task_precision']:.3f}")
print(f"Total Tool Calls:      {precision_metrics['total_tool_calls']}")
print(f"Correct Tool Calls:    {precision_metrics['correct_tool_calls']}")

## 9. Tool Invocation Recall

### Definition
**Tool Invocation Recall** measures the proportion of expected tools that were actually called.

### Formula
```
Recall = correct_tool_calls / tool_calls_expected
```

This metric captures whether the agent is calling all the tools it should.

In [ ]:
def compute_tool_invocation_recall(results: List[TaskEvaluation]) -> Dict[str, Any]:
    """
    Compute Tool Invocation Recall across all tasks.
    
    Recall = tools_correctly_called / tools_expected
    
    Measures whether the agent calls all expected tools.
    """
    total_expected = 0
    total_recalled = 0
    per_task_recall = []
    
    for result in results:
        expected_set = set(result.expected_tools)
        actual_set = set(result.actual_tools)
        
        # Also check acceptable alternatives
        # If an alternative set was used instead, count as recall success
        best_recall = 0
        all_tool_sets = [expected_set] + [set(alt) for alt in result.acceptable_alternatives]
        
        for tool_set in all_tool_sets:
            if len(tool_set) > 0:
                recalled = len(actual_set & tool_set)
                recall_rate = recalled / len(tool_set)
                best_recall = max(best_recall, recall_rate)
        
        task_expected = len(expected_set)
        task_recalled = len(actual_set & expected_set)
        
        total_expected += task_expected
        total_recalled += task_recalled
        
        # Per-task recall (use best recall if alternatives exist)
        if task_expected > 0:
            per_task_recall.append(best_recall if result.acceptable_alternatives else task_recalled / task_expected)
        else:
            # No tools expected - recall is 1.0 if none called, 0.0 otherwise
            per_task_recall.append(1.0 if len(result.actual_tools) == 0 else 0.0)
    
    # Overall recall
    overall_recall = total_recalled / total_expected if total_expected > 0 else 1.0
    avg_task_recall = sum(per_task_recall) / len(per_task_recall) if per_task_recall else 0.0
    
    return {
        "overall_recall": overall_recall,
        "avg_task_recall": avg_task_recall,
        "total_expected": total_expected,
        "total_recalled": total_recalled,
        "per_task_recall": per_task_recall
    }


# Compute and display
recall_metrics = compute_tool_invocation_recall(evaluation_results)

print("=" * 50)
print("TOOL INVOCATION RECALL")
print("=" * 50)
print(f"Overall Recall:        {recall_metrics['overall_recall']:.3f}")
print(f"Avg Task Recall:       {recall_metrics['avg_task_recall']:.3f}")
print(f"Total Expected:        {recall_metrics['total_expected']}")
print(f"Total Recalled:        {recall_metrics['total_recalled']}")

## 10. Average Tool Calls per Task

### Definition
**Average Tool Calls per Task** measures the mean number of tool invocations per task, with efficiency classification.

### Efficiency Classification
- **Too Few**: actual_calls < optimal_calls
- **Optimal**: actual_calls == optimal_calls
- **Too Many**: actual_calls > optimal_calls

In [ ]:
def compute_avg_tool_calls_per_task(results: List[TaskEvaluation]) -> Dict[str, Any]:
    """
    Compute average tool calls per task and efficiency classification.
    
    Classifies each task as:
    - Too Few: called fewer tools than optimal
    - Optimal: called exact number of tools needed
    - Too Many: called more tools than optimal
    """
    total_calls = 0
    efficiency_counts = {"too_few": 0, "optimal": 0, "too_many": 0}
    per_task_efficiency = []
    calls_per_task = []
    
    for result in results:
        actual_count = len(result.actual_tools)
        optimal_count = result.optimal_tool_count
        
        total_calls += actual_count
        calls_per_task.append(actual_count)
        
        # Classify efficiency
        if actual_count < optimal_count:
            efficiency = "too_few"
        elif actual_count == optimal_count:
            efficiency = "optimal"
        else:
            efficiency = "too_many"
        
        efficiency_counts[efficiency] += 1
        per_task_efficiency.append(efficiency)
    
    avg_calls = total_calls / len(results) if results else 0
    
    return {
        "avg_calls_per_task": avg_calls,
        "total_calls": total_calls,
        "efficiency_counts": efficiency_counts,
        "efficiency_rate": {
            k: v / len(results) if results else 0 
            for k, v in efficiency_counts.items()
        },
        "per_task_efficiency": per_task_efficiency,
        "calls_per_task": calls_per_task
    }


# Compute and display
avg_calls_metrics = compute_avg_tool_calls_per_task(evaluation_results)

print("=" * 50)
print("AVERAGE TOOL CALLS PER TASK")
print("=" * 50)
print(f"Average Calls/Task:    {avg_calls_metrics['avg_calls_per_task']:.2f}")
print(f"Total Tool Calls:      {avg_calls_metrics['total_calls']}")
print(f"\nEfficiency Distribution:")
print(f"  Too Few:   {avg_calls_metrics['efficiency_counts']['too_few']} ({avg_calls_metrics['efficiency_rate']['too_few']*100:.1f}%)")
print(f"  Optimal:   {avg_calls_metrics['efficiency_counts']['optimal']} ({avg_calls_metrics['efficiency_rate']['optimal']*100:.1f}%)")
print(f"  Too Many:  {avg_calls_metrics['efficiency_counts']['too_many']} ({avg_calls_metrics['efficiency_rate']['too_many']*100:.1f}%)")

## 11. Tool Success Rate

### Definition
**Tool Success Rate** measures the proportion of tool calls that executed successfully (without errors).

### Formula
```
Success Rate = successful_tool_calls / total_tool_calls
```

A tool call is considered "successful" if its output does not contain error messages.

In [ ]:
def compute_tool_success_rate(results: List[TaskEvaluation]) -> Dict[str, Any]:
    """
    Compute Tool Success Rate across all tasks.
    
    Success Rate = successful_tool_calls / total_tool_calls
    
    A tool call is "successful" if it doesn't return an error.
    """
    total_calls = 0
    successful_calls = 0
    failed_calls = 0
    total_retries = 0
    per_task_success_rate = []
    
    for result in results:
        task_total = len(result.tool_call_logs)
        task_successful = sum(1 for log in result.tool_call_logs if log.success)
        task_retries = sum(log.retry_count for log in result.tool_call_logs)
        
        total_calls += task_total
        successful_calls += task_successful
        failed_calls += (task_total - task_successful)
        total_retries += task_retries
        
        # Per-task success rate
        if task_total > 0:
            per_task_success_rate.append(task_successful / task_total)
        else:
            per_task_success_rate.append(1.0)  # No calls = no failures
    
    overall_success_rate = successful_calls / total_calls if total_calls > 0 else 1.0
    avg_task_success_rate = sum(per_task_success_rate) / len(per_task_success_rate) if per_task_success_rate else 1.0
    
    return {
        "overall_success_rate": overall_success_rate,
        "avg_task_success_rate": avg_task_success_rate,
        "total_calls": total_calls,
        "successful_calls": successful_calls,
        "failed_calls": failed_calls,
        "total_retries": total_retries,
        "per_task_success_rate": per_task_success_rate
    }


# Compute and display
success_metrics = compute_tool_success_rate(evaluation_results)

print("=" * 50)
print("TOOL SUCCESS RATE")
print("=" * 50)
print(f"Overall Success Rate:  {success_metrics['overall_success_rate']:.3f}")
print(f"Avg Task Success Rate: {success_metrics['avg_task_success_rate']:.3f}")
print(f"Total Calls:           {success_metrics['total_calls']}")
print(f"Successful Calls:      {success_metrics['successful_calls']}")
print(f"Failed Calls:          {success_metrics['failed_calls']}")
print(f"Total Retries:         {success_metrics['total_retries']}")

## 12. Tool-Attributable Task Success

### Definition
**Tool-Attributable Task Success** measures whether task success can be attributed to tool usage.

### Criteria
A task has tool-attributable success if:
1. The task required external data (from tools)
2. Tools were called
3. Tool outputs appear in the final response

This helps distinguish between:
- Tasks where tools genuinely contributed to the answer
- Tasks where the LLM could have answered without tools

In [ ]:
def compute_tool_attributable_success(results: List[TaskEvaluation]) -> Dict[str, Any]:
    """
    Compute Tool-Attributable Task Success.
    
    Measures whether task success can be attributed to tool usage:
    - Task required external data
    - Tools were actually called
    - Tool outputs appear in final response
    """
    total_tasks = len(results)
    tasks_requiring_tools = 0
    tool_attributable_successes = 0
    per_task_attribution = []
    
    for result in results:
        attribution_info = {
            "requires_external_data": result.requires_external_data,
            "tools_called": len(result.tool_call_logs) > 0,
            "output_uses_tool_data": False,
            "is_attributable": False
        }
        
        if result.requires_external_data:
            tasks_requiring_tools += 1
            
            if result.tool_call_logs and result.output:
                # Check if tool output appears in response
                output_uses_tool_data = check_tool_output_in_response(
                    result.tool_call_logs, 
                    result.output
                )
                attribution_info["output_uses_tool_data"] = output_uses_tool_data
                
                if output_uses_tool_data:
                    tool_attributable_successes += 1
                    attribution_info["is_attributable"] = True
        
        per_task_attribution.append(attribution_info)
    
    # Rates
    attribution_rate = tool_attributable_successes / tasks_requiring_tools if tasks_requiring_tools > 0 else 0.0
    
    return {
        "total_tasks": total_tasks,
        "tasks_requiring_tools": tasks_requiring_tools,
        "tool_attributable_successes": tool_attributable_successes,
        "attribution_rate": attribution_rate,
        "per_task_attribution": per_task_attribution
    }


# Compute and display
attribution_metrics = compute_tool_attributable_success(evaluation_results)

print("=" * 50)
print("TOOL-ATTRIBUTABLE TASK SUCCESS")
print("=" * 50)
print(f"Total Tasks:              {attribution_metrics['total_tasks']}")
print(f"Tasks Requiring Tools:    {attribution_metrics['tasks_requiring_tools']}")
print(f"Attributable Successes:   {attribution_metrics['tool_attributable_successes']}")
print(f"Attribution Rate:         {attribution_metrics['attribution_rate']:.3f}")

## 13. Cost per Successful Task

### Definition
**Cost per Successful Task** measures the average cost incurred for each successfully completed task.

### Formula
```
Total Cost = LLM Token Cost + Tool Execution Cost
Cost per Success = Total Cost / Number of Successful Tasks
```

### Cost Classification
- **Low**: < $0.001 per task
- **Moderate**: $0.001 - $0.01 per task  
- **High**: > $0.01 per task

In [ ]:
def compute_cost_per_successful_task(results: List[TaskEvaluation]) -> Dict[str, Any]:
    """
    Compute Cost per Successful Task.
    
    Total Cost = LLM tokens + tool execution cost
    Cost per Success = Total Cost / Successful Tasks
    
    Classification:
    - Low: < $0.001
    - Moderate: $0.001 - $0.01
    - High: > $0.01
    """
    total_llm_cost = 0.0
    total_tool_cost = 0.0
    successful_tasks = 0
    per_task_cost = []
    
    for result in results:
        task_cost = result.total_cost
        total_llm_cost += result.llm_cost
        total_tool_cost += result.tool_cost
        per_task_cost.append(task_cost)
        
        if result.task_success:
            successful_tasks += 1
    
    total_cost = total_llm_cost + total_tool_cost
    cost_per_success = total_cost / successful_tasks if successful_tasks > 0 else float('inf')
    avg_cost_per_task = total_cost / len(results) if results else 0.0
    
    # Classify cost
    if cost_per_success < 0.001:
        cost_classification = "Low"
    elif cost_per_success < 0.01:
        cost_classification = "Moderate"
    else:
        cost_classification = "High"
    
    return {
        "total_cost": total_cost,
        "total_llm_cost": total_llm_cost,
        "total_tool_cost": total_tool_cost,
        "successful_tasks": successful_tasks,
        "cost_per_success": cost_per_success,
        "avg_cost_per_task": avg_cost_per_task,
        "cost_classification": cost_classification,
        "per_task_cost": per_task_cost
    }


# Compute and display
cost_metrics = compute_cost_per_successful_task(evaluation_results)

print("=" * 50)
print("COST PER SUCCESSFUL TASK")
print("=" * 50)
print(f"Total Cost:            ${cost_metrics['total_cost']:.6f}")
print(f"  - LLM Cost:          ${cost_metrics['total_llm_cost']:.6f}")
print(f"  - Tool Cost:         ${cost_metrics['total_tool_cost']:.6f}")
print(f"Successful Tasks:      {cost_metrics['successful_tasks']}")
print(f"Cost per Success:      ${cost_metrics['cost_per_success']:.6f}")
print(f"Avg Cost per Task:     ${cost_metrics['avg_cost_per_task']:.6f}")
print(f"Cost Classification:   {cost_metrics['cost_classification']}")

---

## 14. Consolidated Evaluation Summary

This section combines all metrics into a single summary table and report.

In [ ]:
def generate_consolidated_summary(
    results: List[TaskEvaluation],
    precision_metrics: Dict,
    recall_metrics: Dict,
    avg_calls_metrics: Dict,
    success_metrics: Dict,
    attribution_metrics: Dict,
    cost_metrics: Dict
) -> pd.DataFrame:
    """
    Generate a consolidated summary DataFrame with all metrics.
    """
    # Create summary data
    summary_data = [
        # Tool Invocation Metrics
        {"Category": "Tool Invocation", "Metric": "Precision (Overall)", "Value": f"{precision_metrics['overall_precision']:.3f}", "Description": "Correct calls / Total calls"},
        {"Category": "Tool Invocation", "Metric": "Precision (Avg per Task)", "Value": f"{precision_metrics['avg_task_precision']:.3f}", "Description": "Average precision across tasks"},
        {"Category": "Tool Invocation", "Metric": "Recall (Overall)", "Value": f"{recall_metrics['overall_recall']:.3f}", "Description": "Recalled / Expected"},
        {"Category": "Tool Invocation", "Metric": "Recall (Avg per Task)", "Value": f"{recall_metrics['avg_task_recall']:.3f}", "Description": "Average recall across tasks"},
        
        # Efficiency Metrics
        {"Category": "Efficiency", "Metric": "Avg Calls per Task", "Value": f"{avg_calls_metrics['avg_calls_per_task']:.2f}", "Description": "Mean tool invocations"},
        {"Category": "Efficiency", "Metric": "Optimal Call Rate", "Value": f"{avg_calls_metrics['efficiency_rate']['optimal']*100:.1f}%", "Description": "Tasks with optimal tool count"},
        {"Category": "Efficiency", "Metric": "Over-calling Rate", "Value": f"{avg_calls_metrics['efficiency_rate']['too_many']*100:.1f}%", "Description": "Tasks with excess tool calls"},
        {"Category": "Efficiency", "Metric": "Under-calling Rate", "Value": f"{avg_calls_metrics['efficiency_rate']['too_few']*100:.1f}%", "Description": "Tasks with insufficient calls"},
        
        # Success Metrics
        {"Category": "Success", "Metric": "Tool Success Rate", "Value": f"{success_metrics['overall_success_rate']:.3f}", "Description": "Successful / Total calls"},
        {"Category": "Success", "Metric": "Failed Tool Calls", "Value": str(success_metrics['failed_calls']), "Description": "Number of failed executions"},
        {"Category": "Success", "Metric": "Attribution Rate", "Value": f"{attribution_metrics['attribution_rate']:.3f}", "Description": "Tool-attributable successes"},
        
        # Cost Metrics
        {"Category": "Cost", "Metric": "Total Cost", "Value": f"${cost_metrics['total_cost']:.6f}", "Description": "LLM + Tool costs"},
        {"Category": "Cost", "Metric": "Cost per Success", "Value": f"${cost_metrics['cost_per_success']:.6f}", "Description": "Cost per successful task"},
        {"Category": "Cost", "Metric": "Cost Classification", "Value": cost_metrics['cost_classification'], "Description": "Low/Moderate/High"},
    ]
    
    return pd.DataFrame(summary_data)


# Generate consolidated summary
summary_df = generate_consolidated_summary(
    evaluation_results,
    precision_metrics,
    recall_metrics,
    avg_calls_metrics,
    success_metrics,
    attribution_metrics,
    cost_metrics
)

print("\n" + "="*80)
print("CONSOLIDATED EVALUATION SUMMARY")
print("="*80 + "\n")

# Display as formatted table
print(summary_df.to_string(index=False))

In [ ]:
# Display as styled DataFrame (works better in Jupyter/Colab)
summary_df.style.set_properties(**{'text-align': 'left'}).set_table_styles([
    {'selector': 'th', 'props': [('text-align', 'left')]}
])

## 15. Per-Task Detailed Results

In [ ]:
def generate_per_task_summary(results: List[TaskEvaluation]) -> pd.DataFrame:
    """Generate per-task summary DataFrame."""
    rows = []
    
    for i, result in enumerate(results, 1):
        expected_set = set(result.expected_tools)
        actual_set = set(result.actual_tools)
        
        # Calculate per-task metrics
        precision = len(expected_set & actual_set) / len(actual_set) if actual_set else (1.0 if not expected_set else 0.0)
        recall = len(expected_set & actual_set) / len(expected_set) if expected_set else (1.0 if not actual_set else 0.0)
        
        rows.append({
            "#": i,
            "Question": result.question[:40] + "...",
            "Category": result.category,
            "Expected": ", ".join(result.expected_tools) or "None",
            "Actual": ", ".join(result.actual_tools) or "None",
            "Precision": f"{precision:.2f}",
            "Recall": f"{recall:.2f}",
            "Success": "Yes" if result.task_success else "No",
            "Cost": f"${result.total_cost:.5f}"
        })
    
    return pd.DataFrame(rows)


# Generate and display per-task summary
per_task_df = generate_per_task_summary(evaluation_results)

print("\n" + "="*100)
print("PER-TASK DETAILED RESULTS")
print("="*100 + "\n")

print(per_task_df.to_string(index=False))

## 16. Visualizations

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Precision vs Recall by Category
categories = list(set(r.category for r in evaluation_results))
cat_precision = []
cat_recall = []

for cat in categories:
    cat_results = [r for r in evaluation_results if r.category == cat]
    prec_vals = precision_metrics['per_task_precision']
    rec_vals = recall_metrics['per_task_recall']
    indices = [i for i, r in enumerate(evaluation_results) if r.category == cat]
    cat_precision.append(np.mean([prec_vals[i] for i in indices]))
    cat_recall.append(np.mean([rec_vals[i] for i in indices]))

x = np.arange(len(categories))
width = 0.35
axes[0, 0].bar(x - width/2, cat_precision, width, label='Precision', color='#2196F3')
axes[0, 0].bar(x + width/2, cat_recall, width, label='Recall', color='#4CAF50')
axes[0, 0].set_xlabel('Category')
axes[0, 0].set_ylabel('Score')
axes[0, 0].set_title('Precision vs Recall by Category')
axes[0, 0].set_xticks(x)
axes[0, 0].set_xticklabels(categories, rotation=45, ha='right')
axes[0, 0].legend()
axes[0, 0].set_ylim(0, 1.1)
axes[0, 0].grid(axis='y', alpha=0.3)

# Plot 2: Efficiency Distribution
efficiency_labels = ['Too Few', 'Optimal', 'Too Many']
efficiency_values = [
    avg_calls_metrics['efficiency_counts']['too_few'],
    avg_calls_metrics['efficiency_counts']['optimal'],
    avg_calls_metrics['efficiency_counts']['too_many']
]
colors = ['#f44336', '#4CAF50', '#FF9800']
axes[0, 1].pie(efficiency_values, labels=efficiency_labels, autopct='%1.1f%%', colors=colors, startangle=90)
axes[0, 1].set_title('Tool Call Efficiency Distribution')

# Plot 3: Success Metrics
metrics_names = ['Tool\nSuccess', 'Attribution\nRate', 'Task\nSuccess']
metrics_values = [
    success_metrics['overall_success_rate'],
    attribution_metrics['attribution_rate'],
    cost_metrics['successful_tasks'] / len(evaluation_results)
]
bars = axes[1, 0].bar(metrics_names, metrics_values, color=['#2196F3', '#9C27B0', '#4CAF50'])
axes[1, 0].set_ylabel('Rate')
axes[1, 0].set_title('Success Metrics Overview')
axes[1, 0].set_ylim(0, 1.1)
axes[1, 0].grid(axis='y', alpha=0.3)
for bar, val in zip(bars, metrics_values):
    axes[1, 0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f'{val:.2f}', ha='center')

# Plot 4: Cost Breakdown
cost_labels = ['LLM Cost', 'Tool Cost']
cost_values = [cost_metrics['total_llm_cost'], cost_metrics['total_tool_cost']]
axes[1, 1].bar(cost_labels, cost_values, color=['#3F51B5', '#009688'])
axes[1, 1].set_ylabel('Cost ($)')
axes[1, 1].set_title(f'Cost Breakdown (Total: ${cost_metrics["total_cost"]:.5f})')
axes[1, 1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\nCharts displayed above!")

## 17. Export Results

In [ ]:
import json

# Compile all metrics for export
export_data = {
    "evaluation_summary": {
        "tool_invocation_precision": {
            "overall": precision_metrics['overall_precision'],
            "avg_per_task": precision_metrics['avg_task_precision'],
            "total_calls": precision_metrics['total_tool_calls'],
            "correct_calls": precision_metrics['correct_tool_calls']
        },
        "tool_invocation_recall": {
            "overall": recall_metrics['overall_recall'],
            "avg_per_task": recall_metrics['avg_task_recall'],
            "total_expected": recall_metrics['total_expected'],
            "total_recalled": recall_metrics['total_recalled']
        },
        "avg_tool_calls_per_task": {
            "average": avg_calls_metrics['avg_calls_per_task'],
            "efficiency_distribution": avg_calls_metrics['efficiency_counts']
        },
        "tool_success_rate": {
            "overall": success_metrics['overall_success_rate'],
            "successful_calls": success_metrics['successful_calls'],
            "failed_calls": success_metrics['failed_calls']
        },
        "tool_attributable_success": {
            "attribution_rate": attribution_metrics['attribution_rate'],
            "attributable_successes": attribution_metrics['tool_attributable_successes']
        },
        "cost_per_successful_task": {
            "total_cost": cost_metrics['total_cost'],
            "cost_per_success": cost_metrics['cost_per_success'],
            "classification": cost_metrics['cost_classification']
        }
    },
    "per_task_results": [
        {
            "question": r.question,
            "category": r.category,
            "expected_tools": r.expected_tools,
            "actual_tools": r.actual_tools,
            "task_success": r.task_success,
            "cost": r.total_cost
        }
        for r in evaluation_results
    ]
}

# Save to JSON
with open("tool_eval_v2_results.json", "w") as f:
    json.dump(export_data, f, indent=2)

# Save summary table to CSV
summary_df.to_csv("tool_eval_v2_summary.csv", index=False)
per_task_df.to_csv("tool_eval_v2_per_task.csv", index=False)

print("Results exported to:")
print("  - tool_eval_v2_results.json")
print("  - tool_eval_v2_summary.csv")
print("  - tool_eval_v2_per_task.csv")

# For Colab: Download files
try:
    from google.colab import files
    files.download('tool_eval_v2_results.json')
    files.download('tool_eval_v2_summary.csv')
    print("\nDownload started!")
except ImportError:
    print("\n(Not running in Colab - files saved locally)")

---

## Summary of v2 Metrics

| Metric | Definition | Formula |
|--------|------------|--------|
| **Tool Invocation Precision** | Proportion of tool calls that were correct | correct_calls / total_calls |
| **Tool Invocation Recall** | Proportion of expected tools that were called | recalled / expected |
| **Avg Tool Calls per Task** | Mean number of tool invocations | total_calls / num_tasks |
| **Tool Success Rate** | Proportion of tool calls without errors | successful / total_calls |
| **Tool-Attributable Success** | Tasks where tools contributed to answer | attribution_count / tool_required_tasks |
| **Cost per Successful Task** | Average cost per successful completion | total_cost / successful_tasks |

### Key Insights to Look For:
1. **High Precision, Low Recall**: Agent is conservative, misses needed tools
2. **Low Precision, High Recall**: Agent over-calls tools
3. **Low Attribution Rate**: Tools not contributing to final answers
4. **High Cost Classification**: Consider optimizing token usage or tool selection